In [1]:
import pandas as pd

In [2]:
# 1. LOAD DATASET
df = pd.read_csv("india_portfolio_dirty.csv")

print("Original Shape:", df.shape)

Original Shape: (5080, 27)


In [3]:
# 2. REMOVE COMPLETELY BLANK COLUMNS
df = df.dropna(axis=1, how="all")

In [4]:
# 3. REMOVE EXACT DUPLICATE ROWS
df = df.drop_duplicates()

In [5]:
# 4. CLEAN COLUMN NAMES
df.columns = df.columns.str.strip()

In [6]:
# 5. CLEAN TEXT COLUMNS
text_columns = [
    "trade_id",
    "ticker",
    "company_name",
    "exchange",
    "sector",
    "portfolio",
    "analyst_rating"
]

for col in text_columns:
    if col in df.columns:
        df[col] = df[col].astype("string").str.strip()

In [7]:
# 6. CLEAN TRADE ID WHITESPACE
if "trade_id" in df.columns:
    df["trade_id"] = (
        df["trade_id"]
        .str.replace(r"\s+", "", regex=True)
        .str.strip()
    )

In [8]:
# 7. STANDARDIZE TICKER
if "ticker" in df.columns:
    df["ticker"] = df["ticker"].str.upper().str.strip()

In [9]:
# 8. STANDARDIZE EXCHANGE
if "exchange" in df.columns:

    df["exchange"] = (
        df["exchange"]
        .str.upper()
        .str.strip()
    )

    exchange_map = {
        "N": "NSE",
        "NSE": "NSE",
        "NSE ": "NSE",
        "BSE": "BSE",
        "B": "BSE"
    }

    df["exchange"] = df["exchange"].replace(exchange_map)

In [14]:
# 9. STANDARDIZE SECTOR
if "sector" in df.columns:

    df["sector"] = (
        df["sector"]
        .str.upper()
        .str.strip()
    )

    sector_map = {
        "TECH": "TECHNOLOGY",
        "TECHNOLOGY": "TECHNOLOGY",
        "IT": "TECHNOLOGY",

        "BANKING": "FINANCE",
        "BANK": "FINANCE",
        "FINANCE": "FINANCE",
        "FINANCIAL": "FINANCE",

        "HEALTH": "HEALTHCARE",
        "HEALTHCARE": "HEALTHCARE",

        "PHARMA": "PHARMACEUTICALS",
        "PHARMACEUTICAL": "PHARMACEUTICALS",
        "PHARMACEUTICALS": "PHARMACEUTICALS",

        "AUTO": "AUTOMOBILE",
        "AUTOMOBILE": "AUTOMOBILE",

        "ENERGY": "ENERGY",

        "CONSUMER": "CONSUMER",

        "TELECOM": "TELECOMMUNICATION",
    "TELECOMMUNICATION": "TELECOMMUNICATION"
    }

    df["sector"] = df["sector"].replace(sector_map)

In [15]:
# 10. STANDARDIZE ANALYST RATINGS
if "analyst_rating" in df.columns:

    df["analyst_rating"] = (
        df["analyst_rating"]
        .str.upper()
        .str.strip()
    )

    rating_map = {
        "BUY": "BUY",
        "STRONG BUY": "BUY",
        "BUY ": "BUY",

        "HOLD": "HOLD",
        "NEUTRAL": "HOLD",

        "SELL": "SELL",
        "STRONG SELL": "SELL",

        "N/A": pd.NA,
        "NA": pd.NA,
        "--": pd.NA,
        "-": pd.NA,
        "": pd.NA
    }

    df["analyst_rating"] = df["analyst_rating"].replace(rating_map)

In [16]:
# 11. STANDARDIZE PORTFOLIO
# ------------------------------------------------------------

if "portfolio" in df.columns:

    df["portfolio"] = (
        df["portfolio"]
        .astype("string")
        .str.upper()
        .str.strip()
    )

In [17]:
# 12. CLEAN DATE
# ------------------------------------------------------------

if "trade_date" in df.columns:

    df["trade_date"] = (
        df["trade_date"]
        .astype("string")
        .str.strip()
    )

    df["trade_date"] = pd.to_datetime(
        df["trade_date"],
        errors="coerce",
        dayfirst=True
    )

In [18]:
# 13. NUMERIC COLUMNS
numeric_columns = [
    "open_price_inr",
    "high_price_inr",
    "low_price_inr",
    "close_price_inr",
    "prev_close_inr",
    "volume",
    "market_cap_cr",
    "pe_ratio",
    "pb_ratio",
    "dividend_yield",
    "beta",
    "52w_high_inr",
    "52w_low_inr",
    "nifty50_weight",
    "shares_held",
    "purchase_price_inr"
]

for col in numeric_columns:

    if col in df.columns:

        df[col] = (
            df[col]
            .astype("string")
            .str.replace(",", "", regex=False)
            .str.replace("₹", "", regex=False)
            .str.replace("Cr", "", regex=False)
            .str.strip()
        )

        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

In [19]:
# 14. CLEAN MARKET CAP
# Example:
# ₹1,23,456 Cr → 123456
# ------------------------------------------------------------

if "market_cap_cr" in df.columns:

    df["market_cap_cr"] = (
        df["market_cap_cr"]
        .astype("string")
        .str.replace("₹", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.replace("Cr", "", regex=False)
        .str.strip()
    )

    df["market_cap_cr"] = pd.to_numeric(
        df["market_cap_cr"],
        errors="coerce"
    )

In [20]:
# 15. INVALID PRICE VALUES
# Negative and zero prices are invalid
# ------------------------------------------------------------

price_columns = [
    "open_price_inr",
    "high_price_inr",
    "low_price_inr",
    "close_price_inr",
    "prev_close_inr",
    "purchase_price_inr",
    "52w_high_inr",
    "52w_low_inr"
]

for col in price_columns:

    if col in df.columns:

        df.loc[
            df[col] <= 0,
            col
        ] = pd.NA

In [21]:
# 16. INVALID PE / PB VALUES
for col in ["pe_ratio", "pb_ratio"]:

    if col in df.columns:

        df.loc[
            df[col] <= 0,
            col
        ] = pd.NA

        df.loc[
            df[col] > 1000,
            col
        ] = pd.NA

In [22]:
# 17. INVALID VOLUME
# ------------------------------------------------------------

if "volume" in df.columns:

    df.loc[
        df["volume"] <= 0,
        "volume"
    ] = pd.NA

In [23]:
# 18. INVALID SHARES HELD
if "shares_held" in df.columns:

    df.loc[
        df["shares_held"] < 0,
        "shares_held"
    ] = pd.NA

In [24]:
# 19. INVALID BETA
if "beta" in df.columns:

    df.loc[
        (df["beta"] < -10) | (df["beta"] > 10),
        "beta"
    ] = pd.NA

In [25]:
# 20. INVALID DIVIDEND YIELD
if "dividend_yield" in df.columns:

    df.loc[
        (df["dividend_yield"] < 0) |
        (df["dividend_yield"] > 100),
        "dividend_yield"
    ] = pd.NA

In [26]:
# 21. FIX OHLC USING GROUPED PRICE HISTORY
if "ticker" in df.columns and "trade_date" in df.columns:

    df = df.sort_values(
        ["ticker", "trade_date"]
    ).reset_index(drop=True)

In [27]:
# 22. FIX MISSING PREVIOUS CLOSE
# Use previous day's close for the same ticker
# ------------------------------------------------------------

if "prev_close_inr" in df.columns:

    previous_close = (
        df.groupby("ticker")["close_price_inr"]
        .shift(1)
    )

    df["prev_close_inr"] = (
        df["prev_close_inr"]
        .fillna(previous_close)
    )

In [28]:
# 23. FIX MISSING CLOSE
# Use valid OHLC values
# ------------------------------------------------------------

if "close_price_inr" in df.columns:

    df["close_price_inr"] = (
        df["close_price_inr"]
        .fillna(df["open_price_inr"])
        .fillna(df["prev_close_inr"])
    )

In [29]:
# 24. FIX MISSING OPEN
# ------------------------------------------------------------

if "open_price_inr" in df.columns:

    df["open_price_inr"] = (
        df["open_price_inr"]
        .fillna(df["prev_close_inr"])
        .fillna(df["close_price_inr"])
    )

In [30]:
# 25. FIX MISSING HIGH
# High must be >= Open, Close and Low
# ------------------------------------------------------------

if "high_price_inr" in df.columns:

    df["high_price_inr"] = (
        df[
            [
                "high_price_inr",
                "open_price_inr",
                "close_price_inr",
                "low_price_inr"
            ]
        ]
        .max(axis=1, skipna=True)
    )

In [31]:
# 26. FIX MISSING LOW
# Low must be <= Open, Close and High
# ------------------------------------------------------------

if "low_price_inr" in df.columns:

    df["low_price_inr"] = (
        df[
            [
                "low_price_inr",
                "open_price_inr",
                "close_price_inr"
            ]
        ]
        .min(axis=1, skipna=True)
    )

In [32]:
# 27. FIX OHLC RELATIONSHIPS
# ------------------------------------------------------------

if all(
    col in df.columns
    for col in [
        "open_price_inr",
        "high_price_inr",
        "low_price_inr",
        "close_price_inr"
    ]
):

    df["high_price_inr"] = (
        df[
            [
                "high_price_inr",
                "open_price_inr",
                "close_price_inr"
            ]
        ]
        .max(axis=1, skipna=True)
    )

    df["low_price_inr"] = (
        df[
            [
                "low_price_inr",
                "open_price_inr",
                "close_price_inr"
            ]
        ]
        .min(axis=1, skipna=True)
    )

In [33]:
# 28. FIX 52-WEEK HIGH
# 52W High must be >= Current Price
# ------------------------------------------------------------

if "52w_high_inr" in df.columns:

    df["52w_high_inr"] = (
        df[
            [
                "52w_high_inr",
                "close_price_inr"
            ]
        ]
        .max(axis=1, skipna=True)
    )

In [34]:
# 29. FIX 52-WEEK LOW
# 52W Low must be <= Current Price
# ------------------------------------------------------------

if "52w_low_inr" in df.columns:

    df["52w_low_inr"] = (
        df[
            [
                "52w_low_inr",
                "close_price_inr"
            ]
        ]
        .min(axis=1, skipna=True)
    )

In [35]:
# 30. FIX COMPANY NAME FROM TICKER
# Use the most common valid company name
# ------------------------------------------------------------

if "ticker" in df.columns and "company_name" in df.columns:

    company_map = (
        df.dropna(subset=["company_name"])
        .groupby("ticker")["company_name"]
        .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else pd.NA)
    )

    df["company_name"] = (
        df["company_name"]
        .fillna(df["ticker"].map(company_map))
    )

In [36]:
# 31. FIX SECTOR FROM TICKER
# ------------------------------------------------------------

if "ticker" in df.columns and "sector" in df.columns:

    sector_map_from_data = (
        df.dropna(subset=["sector"])
        .groupby("ticker")["sector"]
        .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else pd.NA)
    )

    df["sector"] = (
        df["sector"]
        .fillna(df["ticker"].map(sector_map_from_data))
    )

In [37]:
# 32. FIX EXCHANGE FROM TICKER
# ------------------------------------------------------------

if "ticker" in df.columns and "exchange" in df.columns:

    exchange_map_from_data = (
        df.dropna(subset=["exchange"])
        .groupby("ticker")["exchange"]
        .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else pd.NA)
    )

    df["exchange"] = (
        df["exchange"]
        .fillna(df["ticker"].map(exchange_map_from_data))
    )

In [38]:
# 33. FIX PORTFOLIO FROM TICKER
# ------------------------------------------------------------

if "ticker" in df.columns and "portfolio" in df.columns:

    portfolio_map = (
        df.dropna(subset=["portfolio"])
        .groupby("ticker")["portfolio"]
        .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else pd.NA)
    )

    df["portfolio"] = (
        df["portfolio"]
        .fillna(df["ticker"].map(portfolio_map))
    )

In [39]:
# 34. FIX NIFTY50 WEIGHT FROM TICKER
# ------------------------------------------------------------

if "ticker" in df.columns and "nifty50_weight" in df.columns:

    weight_map = (
        df.dropna(subset=["nifty50_weight"])
        .groupby("ticker")["nifty50_weight"]
        .median()
    )

    df["nifty50_weight"] = (
        df["nifty50_weight"]
        .fillna(df["ticker"].map(weight_map))
    )

In [40]:
# 35. FIX MARKET CAP FROM TICKER
# ------------------------------------------------------------

if "ticker" in df.columns and "market_cap_cr" in df.columns:

    market_cap_map = (
        df.dropna(subset=["market_cap_cr"])
        .groupby("ticker")["market_cap_cr"]
        .median()
    )

    df["market_cap_cr"] = (
        df["market_cap_cr"]
        .fillna(df["ticker"].map(market_cap_map))
    )

In [41]:
# 36. FIX PE RATIO FROM TICKER
# ------------------------------------------------------------

if "ticker" in df.columns and "pe_ratio" in df.columns:

    pe_map = (
        df.dropna(subset=["pe_ratio"])
        .groupby("ticker")["pe_ratio"]
        .median()
    )

    df["pe_ratio"] = (
        df["pe_ratio"]
        .fillna(df["ticker"].map(pe_map))
    )

In [42]:
# 37. FIX PB RATIO FROM TICKER
# ------------------------------------------------------------

if "ticker" in df.columns and "pb_ratio" in df.columns:

    pb_map = (
        df.dropna(subset=["pb_ratio"])
        .groupby("ticker")["pb_ratio"]
        .median()
    )

    df["pb_ratio"] = (
        df["pb_ratio"]
        .fillna(df["ticker"].map(pb_map))
    )

In [43]:
# 38. FIX BETA FROM TICKER
# ------------------------------------------------------------

if "ticker" in df.columns and "beta" in df.columns:

    beta_map = (
        df.dropna(subset=["beta"])
        .groupby("ticker")["beta"]
        .median()
    )

    df["beta"] = (
        df["beta"]
        .fillna(df["ticker"].map(beta_map))
    )

In [44]:
# 39. FIX DIVIDEND YIELD FROM TICKER
# ------------------------------------------------------------

if "ticker" in df.columns and "dividend_yield" in df.columns:

    dividend_map = (
        df.dropna(subset=["dividend_yield"])
        .groupby("ticker")["dividend_yield"]
        .median()
    )

    df["dividend_yield"] = (
        df["dividend_yield"]
        .fillna(df["ticker"].map(dividend_map))
    )

In [45]:
# 40. FIX PURCHASE PRICE FROM TICKER
# ------------------------------------------------------------

if "ticker" in df.columns and "purchase_price_inr" in df.columns:

    purchase_map = (
        df.dropna(subset=["purchase_price_inr"])
        .groupby("ticker")["purchase_price_inr"]
        .median()
    )

    df["purchase_price_inr"] = (
        df["purchase_price_inr"]
        .fillna(df["ticker"].map(purchase_map))
    )

In [46]:
# 41. FIX SHARES HELD
# ------------------------------------------------------------

if "ticker" in df.columns and "shares_held" in df.columns:

    shares_map = (
        df.dropna(subset=["shares_held"])
        .groupby("ticker")["shares_held"]
        .median()
    )

    df["shares_held"] = (
        df["shares_held"]
        .fillna(df["ticker"].map(shares_map))
    )

In [47]:
# 42. FIX VOLUME
# Use ticker median
# ------------------------------------------------------------

if "ticker" in df.columns and "volume" in df.columns:

    volume_map = (
        df.dropna(subset=["volume"])
        .groupby("ticker")["volume"]
        .median()
    )

    df["volume"] = (
        df["volume"]
        .fillna(df["ticker"].map(volume_map))
    )

In [48]:
# 43. SORT DATA
# ------------------------------------------------------------

sort_columns = [
    col for col in ["ticker", "trade_date"]
    if col in df.columns
]

if sort_columns:
    df = df.sort_values(
        sort_columns
    ).reset_index(drop=True)

In [49]:
# 44. FINAL TEXT CLEANING
# ------------------------------------------------------------

for col in df.select_dtypes(include=["string", "object"]).columns:

    df[col] = (
        df[col]
        .astype("string")
        .str.strip()
    )

In [50]:
# 45. FINAL DUPLICATE CHECK
# ------------------------------------------------------------

df = df.drop_duplicates().reset_index(drop=True)

In [51]:
# 46. FINAL DATA TYPE CONVERSION
# ------------------------------------------------------------

for col in numeric_columns:

    if col in df.columns:
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

if "trade_date" in df.columns:

    df["trade_date"] = pd.to_datetime(
        df["trade_date"],
        errors="coerce"
    )

In [52]:
# 47. FINAL VALIDATION
# ------------------------------------------------------------

print("\n==============================")
print("FINAL DATASET REPORT")
print("==============================")

print("Final Shape:", df.shape)

print("\nDuplicate Rows:")
print(df.duplicated().sum())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nData Types:")
print(df.dtypes)

print("\nTicker Values:")
print(df["ticker"].dropna().unique())

print("\nExchange Values:")
print(df["exchange"].dropna().unique())

print("\nSector Values:")
print(df["sector"].dropna().unique())

print("\nAnalyst Ratings:")
print(df["analyst_rating"].dropna().unique())


FINAL DATASET REPORT
Final Shape: (5019, 25)

Duplicate Rows:
0

Missing Values:
trade_id                 0
ticker                   0
company_name             0
exchange                 0
sector                   0
trade_date            3079
open_price_inr           0
high_price_inr           0
low_price_inr            0
close_price_inr          0
prev_close_inr           0
volume                   0
market_cap_cr            0
pe_ratio                 0
pb_ratio                 0
dividend_yield           0
beta                     0
52w_high_inr             0
52w_low_inr              0
nifty50_weight           0
portfolio                0
analyst_rating         340
shares_held              0
purchase_price_inr       0
currency                 0
dtype: int64

Data Types:
trade_id                      string
ticker                        string
company_name                  string
exchange                      string
sector                        string
trade_date            datetime64

In [54]:
rating_map = {
    "SB": "BUY",
    "STRONG BUY": "BUY",
    "UNDERPERFORM": "SELL",
    "STRONG SELL": "SELL",
    "NEUTRAL": "HOLD",
    "N/A": pd.NA,
    "NA": pd.NA,
    "--": pd.NA,
    "-": pd.NA
}

df["analyst_rating"] = (
    df["analyst_rating"]
    .astype("string")
    .str.upper()
    .str.strip()
    .replace(rating_map)
)

In [55]:
rating_map = (
    df.dropna(subset=["analyst_rating"])
      .groupby("ticker")["analyst_rating"]
      .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else pd.NA)
)

df["analyst_rating"] = (
    df["analyst_rating"]
    .fillna(df["ticker"].map(rating_map))
)

In [57]:
raw = pd.read_csv("india_portfolio_dirty.csv")

raw["trade_date"] = (
    raw["trade_date"]
    .astype("string")
    .str.strip()
)

raw["trade_date"] = pd.to_datetime(
    raw["trade_date"],
    errors="coerce",
    dayfirst=True
)

In [58]:
print(
    raw["trade_date"].isna().sum()
)

3118


In [59]:
print("Duplicate Rows:", df.duplicated().sum())
print(df.isnull().sum())
print(df.dtypes)

Duplicate Rows: 2
trade_id                 0
ticker                   0
company_name             0
exchange                 0
sector                   0
trade_date            3079
open_price_inr           0
high_price_inr           0
low_price_inr            0
close_price_inr          0
prev_close_inr           0
volume                   0
market_cap_cr            0
pe_ratio                 0
pb_ratio                 0
dividend_yield           0
beta                     0
52w_high_inr             0
52w_low_inr              0
nifty50_weight           0
portfolio                0
analyst_rating           0
shares_held              0
purchase_price_inr       0
currency                 0
dtype: int64
trade_id                      string
ticker                        string
company_name                  string
exchange                      string
sector                        string
trade_date            datetime64[us]
open_price_inr               Float64
high_price_inr               Float6

In [60]:
raw["trade_date"] = (
    raw["trade_date"]
    .astype("string")
    .str.strip()
)

raw["trade_date"] = pd.to_datetime(
    raw["trade_date"],
    errors="coerce",
    dayfirst=True
)

print("Missing trade_date:", raw["trade_date"].isna().sum())

Missing trade_date: 3118


In [62]:
print("Duplicate Rows:", df.duplicated().sum())
print(df.isnull().sum())
print(df.dtypes)

Duplicate Rows: 2
trade_id                 0
ticker                   0
company_name             0
exchange                 0
sector                   0
trade_date            3079
open_price_inr           0
high_price_inr           0
low_price_inr            0
close_price_inr          0
prev_close_inr           0
volume                   0
market_cap_cr            0
pe_ratio                 0
pb_ratio                 0
dividend_yield           0
beta                     0
52w_high_inr             0
52w_low_inr              0
nifty50_weight           0
portfolio                0
analyst_rating           0
shares_held              0
purchase_price_inr       0
currency                 0
dtype: int64
trade_id                      string
ticker                        string
company_name                  string
exchange                      string
sector                        string
trade_date            datetime64[us]
open_price_inr               Float64
high_price_inr               Float6

In [63]:
raw["trade_date"] = (
    raw["trade_date"]
    .astype("string")
    .str.strip()
)

raw["trade_date"] = pd.to_datetime(
    raw["trade_date"],
    errors="coerce",
    dayfirst=True
)

print("\nAfter Date Conversion:")
print("Missing trade_date:", raw["trade_date"].isna().sum())


After Date Conversion:
Missing trade_date: 3118


In [64]:
print("Unrecoverable trade dates:", df["trade_date"].isna().sum())

Unrecoverable trade dates: 3079


In [65]:
df = df.drop_duplicates().reset_index(drop=True)

print("Duplicate Rows:", df.duplicated().sum())

Duplicate Rows: 0


In [66]:
# 14. SAVE FINAL DATASET
df.to_csv(
    "perfect_clean_dataset.csv",
    index=False
)

print("\n======================================")
print("FINAL DATASET SAVED")
print("perfect_clean_dataset.csv")
print("======================================")


FINAL DATASET SAVED
perfect_clean_dataset.csv
